# MODIS NDVI Time-Series EDA
Explore MODIS MOD13Q1 16-day composites for the AOI.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import sys; sys.path.insert(0, '..')
from utils.timeseries import gap_fill, savitzky_golay, fit_double_logistic, extract_phenometrics


In [ ]:
# Load pre-fetched MODIS stack (T, H, W, 2)
modis = np.load('../data/raw/modis/modis_stack.npy')
print(f'MODIS stack shape: {modis.shape}')
T, H, W, _ = modis.shape


In [ ]:
# Spatial mean NDVI signal
ndvi_signal = modis[:, :, :, 0].mean(axis=(1, 2))
doys = np.linspace(1, 365, T)

# Gap-fill and smooth
ndvi_filled   = gap_fill(ndvi_signal)
ndvi_smoothed = savitzky_golay(ndvi_filled, window=5, polyorder=3)
ndvi_fitted, params = fit_double_logistic(ndvi_smoothed, doys)
metrics = extract_phenometrics(ndvi_fitted, doys)

print('Phenometrics:', metrics)


In [ ]:
fig, ax = plt.subplots(figsize=(12, 4))
ax.plot(doys, ndvi_signal,   'o', alpha=0.4, label='Raw NDVI')
ax.plot(doys, ndvi_smoothed, '-', lw=2,       label='Smoothed (SG)')
ax.plot(doys, ndvi_fitted,   '--', lw=2,      label='Double Logistic Fit')
ax.axvline(metrics['sos_doy'],  color='green', linestyle=':', label='SOS')
ax.axvline(metrics['peak_doy'], color='gold',  linestyle=':', label='Peak')
ax.axvline(metrics['eos_doy'],  color='red',   linestyle=':', label='EOS')
ax.set_xlabel('Day of Year'); ax.set_ylabel('NDVI'); ax.legend()
ax.set_title('MODIS NDVI Phenology Curve')
plt.tight_layout(); plt.savefig('../data/ndvi_phenology.png', dpi=150)
plt.show()


## Spatial NDVI Maps across time steps

In [ ]:
n_show = min(6, T)
fig, axes = plt.subplots(1, n_show, figsize=(3 * n_show, 3))
for i, ax in enumerate(axes):
    idx = int(i * T / n_show)
    im = ax.imshow(modis[idx, :, :, 0], cmap='RdYlGn', vmin=0, vmax=1)
    ax.set_title(f'DOY {int(doys[idx])}'); ax.axis('off')
plt.colorbar(im, ax=axes[-1], fraction=0.05, label='NDVI')
plt.tight_layout(); plt.show()
